# Convert Depth-Anything-V2-Metric-Indoor-Small → CoreML
Kernel: **Depth Conversion (venv)**. Run cells top to bottom.
Verified working — see `tools/test_full_convert.py`.

In [1]:
# Cell 1 — imports + bicubic patch (coremltools doesn't support bicubic, sub bilinear)
import os, sys, time
import torch
import torch.nn.functional as F

# Guard against double-patching when cell is re-run in same kernel
if not getattr(F.interpolate, '_bicubic_patched', False):
    _orig_interp = F.interpolate
    def _patched_interp(*args, **kwargs):
        if kwargs.get('mode') == 'bicubic':
            kwargs['mode'] = 'bilinear'
        if len(args) >= 4 and args[3] == 'bicubic':
            args = list(args); args[3] = 'bilinear'; args = tuple(args)
        return _orig_interp(*args, **kwargs)
    _patched_interp._bicubic_patched = True
    F.interpolate = _patched_interp
    torch.nn.functional.interpolate = _patched_interp
    torch._C._nn.upsample_bicubic2d = torch._C._nn.upsample_bilinear2d
    print('bicubic -> bilinear patched')
else:
    print('already patched, skipping')

import coremltools as ct
from transformers import AutoModelForDepthEstimation
print('imports done')

Torch version 2.8.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.


bicubic -> bilinear patched


/Users/christianluisefendy/Documents/AIML_Challenge1_Vision/tools/venv_depth/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports done


In [2]:
# Cell 2 — download + load (~500MB first run, cached after)
MODEL_ID = 'depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf'
print(f'loading {MODEL_ID}...')
hf_model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID)
hf_model.eval()
print('model loaded')

loading depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf...
model loaded


In [3]:
# Cell 3 — wrap + trace (normalization baked into model)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class DepthWrapper(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.register_buffer('mean', MEAN)
        self.register_buffer('std',  STD)
    def forward(self, x):
        # x in [0, 1] — CoreML divides input by 255 via the ImageType scale
        x = (x - self.mean) / self.std
        return self.m(pixel_values=x).predicted_depth.unsqueeze(1)

wrapper = DepthWrapper(hf_model)
wrapper.eval()
print('tracing...')
with torch.no_grad():
    traced = torch.jit.trace(wrapper, torch.zeros(1, 3, 518, 518))
print('trace done')

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


tracing...


/Users/christianluisefendy/Documents/AIML_Challenge1_Vision/tools/venv_depth/lib/python3.9/site-packages/transformers/models/dinov2/modeling_dinov2.py:143: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
/Users/christianluisefendy/Documents/AIML_Challenge1_Vision/tools/venv_depth/lib/python3.9/site-packages/transformers/models/depth_anything/modeling_depth_anything.py:159: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if hidden_state.shape != residual.shape:
/Users/christianluisefendy/Documents/AIML_Challenge1_Vision

trace done


In [4]:
# Cell 4 — convert to CoreML
# CPU_ONLY = ~3 sec, runs on CPU (~80ms/frame inference)
# CPU_AND_NE = 5-15 min compile, runs on ANE (~15ms/frame inference)
# Recommend CPU_ONLY first to verify pipeline; recompile with NE later.
COMPUTE_UNITS = ct.ComputeUnit.CPU_ONLY

img_input = ct.ImageType(
    name='image',
    shape=(1, 3, 518, 518),
    scale=1.0/255.0,                 # scalar — per-channel norm is baked into the model
    color_layout=ct.colorlayout.RGB,
)
# Output as grayscale Float16 IMAGE — Swift reads via CVPixelBuffer (OneComponent16Half).
# Matches DepthDriver.depthBufferToU8 / depthBufferToMetricU8 expected format.
print(f'converting ({COMPUTE_UNITS})...')
t0 = time.time()
mlmodel = ct.convert(
    traced,
    inputs=[img_input],
    outputs=[ct.ImageType(name='depth', color_layout=ct.colorlayout.GRAYSCALE_FLOAT16)],
    compute_units=COMPUTE_UNITS,
    convert_to='mlprogram',
    minimum_deployment_target=ct.target.macOS14,
    compute_precision=ct.precision.FLOAT16,
)
print(f'converted in {time.time()-t0:.1f}s')

converting (ComputeUnit.CPU_ONLY)...


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 149.66 passes/s]


converted in 2.2s


In [5]:
# Cell 5 — save into Xcode project folder
OUT_NAME = 'DepthAnythingV2MetricIndoorSmallF16'
OUT_DIR  = os.path.join('..', 'drivingsim', 'drivingsim')
mlmodel.short_description = 'Depth-Anything-V2 Metric Indoor Small — metres (0-20m). Input [0,255] RGB.'
mlmodel.input_description['image']  = '518x518 RGB image, pixel range [0, 255]'
mlmodel.output_description['depth'] = 'Metric depth in metres, Float16, shape (1,1,518,518). Large = far.'
out_path = os.path.join(OUT_DIR, OUT_NAME + '.mlpackage')
mlmodel.save(out_path)
print(f'saved: {os.path.abspath(out_path)}')
print('next: drag the .mlpackage into Xcode → drivingsim target → Build')

saved: /Users/christianluisefendy/Documents/AIML_Challenge1_Vision/drivingsim/drivingsim/DepthAnythingV2MetricIndoorSmallF16.mlpackage
next: drag the .mlpackage into Xcode → drivingsim target → Build
